In [1]:
!nvidia-smi

Tue May 19 14:57:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   73C    P0             30W /   70W |     393MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 1. 구글 드라이브 연동 (마운트)
from google.colab import drive
drive.mount('/content/drive')

import sys
import os
import pandas as pd
import importlib
import inference
# 2. 파이썬 파일들이 있는 드라이브 경로 설정 및 시스템 패스에 추가
project_path = '/content/drive/MyDrive/GCY6902_01_TFT/005.regression_final/'
sys.path.append(project_path)


file_path = os.path.join(project_path, 'data/raw/dataset_8hr_full.parquet')
df = pd.read_parquet(file_path)
df['BAS_DT']=df['open_time'].dt.strftime('%Y%m%d%H')
df = df.rename(columns ={'close':'Close','symbol':'Symbol'})
var_list = [x for x in df.columns if x not in ['date', 'open','symbol_encoded',
                                               'Symbol','open_time', 'target','BAS_DT']] 

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
importlib.reload(inference) # reload 
def profit_simulation(CHOSEN_SEQ, completed_outdir, threshold=0.02):
    # 평가 엔진 초기화
    evaluator = inference.TFTDirectTrajectoryEvaluator(model_dir=completed_outdir)

    # [수정 1] 인퍼런스 엔진의 개편된 3개 아웃풋 스펙에 맞춰 리턴 변수 리시빙 처리
    strategy_report, trajectory_df, xai_dict = evaluator.evaluate_weekly_strategy(
        df_raw=df,
        seq_length=CHOSEN_SEQ,  
        tgt_gap=20,
        base_dt_str='2026040500',   
        target_dt_str='2026041200', 
        threshold=threshold             
    )
    # print(xai_dict) # 필요시 주석 해제하여 확인

    # 전체 요약 성과 출력
    print("📊 [TFT 모형 기반 전종목 스윙 전략 마스터 요약 리포트]")
    display(strategy_report)

    # =========================================================================
    # 3. 포트폴리오 백테스트 성과 분석 및 알파(Alpha) 검증 (롱-숏 양방향 확장형)
    # =========================================================================
    # 포지션 진입 시그널(LONG 또는 SHORT)이 켜진 종목 전체 필터링
    active_portfolio = strategy_report[strategy_report['Strategy_Signal'].isin(['LONG', 'SHORT'])]

    if not active_portfolio.empty:
        portfolio_expected = active_portfolio['Expected_Return_Pct'].mean()
        portfolio_actual_strat = active_portfolio['Actual_Strategy_Return_Pct'].mean()
        
        # 패시브(벤치마크)는 시장에 참여한 전체 종목의 단순 보유 수익률 (고정값)
        portfolio_actual_hold = strategy_report['Passive_Market_Return_Pct'].mean()
        
        print("\n==================================================================")
        print("🎯 [전략 포트폴리오 백테스트 정산 리포트 (Long-Short 통합)]")
        print("==================================================================")
        print(f"🔹 선정 포트폴리오 자산 상황:")
        for _, row in active_portfolio.iterrows():
            print(f"   - {row['Symbol']}: {row['Strategy_Signal']} ({row['Best_Entry_Timing']} ➡️ {row['Best_Exit_Timing']})")
            
        print("------------------------------------------------------------------")
        print(f"📈 모델 궤적상 최적 스윙 예상 수익률     : {portfolio_expected:.2f}%")
        print(f"💰 해당 타이밍 실전 진입 시 실제 수익률  : {portfolio_actual_strat:.2f}%")
        print(f"💤 동기간 전종목 단순 매수(long) 후 방치 수익률 : {portfolio_actual_hold:.2f}%")
        print(f"🚀 양방향 타이밍 최적화를 통한 초과 알파  : {portfolio_actual_strat - portfolio_actual_hold:.2f}%p")
        print("==================================================================\n")
        
        # -----------------------------------------------------------------
        # [Streamlit 시각화 연동용] 최종 아웃풋 파일 자동 백업 로직
        # -----------------------------------------------------------------
        strategy_report.to_excel(completed_outdir + "Backtest_Report_0405_to_0412.xlsx", index=False)
        trajectory_df.to_excel(completed_outdir + "Trajectory_Detail_0405_to_0412.xlsx", index=False)
        
        with open(os.path.join(completed_outdir, 'xai_weights_map.pkl'), 'wb') as f:
            pickle.dump(xai_dict, f)
            
        print(f"💾 대시보드 시각화용 아웃풋 파일 아티팩트 저장 완료 ➡️ 경로: {completed_outdir}")
    else:
        print(f"\n⚠️ 임계치({threshold*100}%)를 넘는 롱/숏 추천 자산이 존재하지 않습니다.")

In [ ]:
# 66
CHOSEN_SEQ = 63  # 21, 42, 63, 84 중 현재 평가할 폴더의 하이퍼파라미터 입력

completed_outdir = project_path + "outputs/dl/66_2605180700_D128_H128_N8_SEQ63/"
for thres in [0.02,0.03,0.05]:
    profit_simulation(CHOSEN_SEQ,completed_outdir,threshold=thres)

/content/drive/MyDrive/GCY6902_01_TFT/005.regression_final/inference.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  
/content/drive/MyDrive/GCY6902_01_TFT/005.regression_final/inference.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  all_symbols = df['Symbol'].unique()


✅ Direct Inference Engine 로드 완료


Evaluating Symbols: 100%|██████████| 5/5 [00:08<00:00,  1.78s/it]

📊 [TFT 모형 기반 전종목 스윙 전략 마스터 요약 리포트]


,Symbol,Current_Price(4/5),Predicted_Entry_Price,Predicted_Exit_Price,Best_Entry_Timing,Best_Exit_Timing,Expected_Return_Pct,Actual_Strategy_Return_Pct,Passive_Market_Return_Pct,Strategy_Signal
0,XRPUSDT,1.311700,1.482907,1.622949,T+1,T+13,9.443797,2.073644,3.209581,LONG
1,DOGEUSDT,0.091740,0.092385,0.094936,T+1,T+13,2.760648,0.305211,1.744064,LONG
2,ETHUSDT,2061.270020,2138.522241,2196.722884,T+1,T+13,2.721536,5.857553,9.525685,LONG
3,BTCUSDT,67177.000000,70934.375381,72457.051106,T+1,T+13,2.146598,5.361954,8.655190,LONG
4,SOLUSDT,80.650002,105.160857,103.682614,T+1,T+21,1.405697,-5.306880,5.306880,HOLD


...

✅ Direct Inference Engine 로드 완료


Evaluating Symbols: 100%|██████████| 5/5 [00:07<00:00,  1.58s/it]

📊 [TFT 모형 기반 전종목 스윙 전략 마스터 요약 리포트]


,Symbol,Current_Price(4/5),Predicted_Entry_Price,Predicted_Exit_Price,Best_Entry_Timing,Best_Exit_Timing,Expected_Return_Pct,Actual_Strategy_Return_Pct,Passive_Market_Return_Pct,Strategy_Signal
0,XRPUSDT,1.311700,1.482907,1.622949,T+1,T+13,9.443797,2.073644,3.209581,LONG
1,DOGEUSDT,0.091740,0.092385,0.094936,T+1,T+13,2.760648,0.305211,1.744064,HOLD
2,ETHUSDT,2061.270020,2138.522241,2196.722884,T+1,T+13,2.721536,5.857553,9.525685,HOLD
3,BTCUSDT,67177.000000,70934.375381,72457.051106,T+1,T+13,2.146598,5.361954,8.655190,HOLD
4,SOLUSDT,80.650002,105.160857,103.682614,T+1,T+21,1.405697,-5.306880,5.306880,HOLD



🎯 [전략 포트폴리오 백테스트 정산 리포트 (Long-Short 통합)]
🔹 선정 포트폴리오 자산 상황:
   - XRPUSDT: LONG (T+1 ➡️ T+13)
------------------------------------------------------------------
📈 모델 궤적상 최적 스윙 예상 수익률     : 9.44%
💰 해당 타이밍 실전 진입 시 실제 수익률  : 2.07%
💤 동기간 전종목 단순 매수(long) 후 방치 수익률 : 5.69%
🚀 양방향 타이밍 최적화를 통한 초과 알파  : -3.61%p

💾 대시보드 시각화용 아웃풋 파일 아티팩트 저장 완료 ➡️ 경로: /content/drive/MyDrive/GCY6902_01_TFT/005.regression_final/outputs/dl/66_2605180700_D128_H128_N8_SEQ63/


In [60]:
# 19
CHOSEN_SEQ = 21  # 21, 42, 63, 84 중 현재 평가할 폴더의 하이퍼파라미터 입력

completed_outdir = project_path + "outputs/dl/19_2605180003_D32_H256_N8_SEQ21/"

for thres in [0.02,0.03,0.04]:
    profit_simulation(CHOSEN_SEQ,completed_outdir,threshold=thres)


/content/drive/MyDrive/GCY6902_01_TFT/005.regression_final/inference.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  
/content/drive/MyDrive/GCY6902_01_TFT/005.regression_final/inference.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  all_symbols = df['Symbol'].unique()


✅ Direct Inference Engine 로드 완료


Evaluating Symbols: 100%|██████████| 5/5 [00:08<00:00,  1.74s/it]

📊 [TFT 모형 기반 전종목 스윙 전략 마스터 요약 리포트]


,Symbol,Current_Price(4/5),Predicted_Entry_Price,Predicted_Exit_Price,Best_Entry_Timing,Best_Exit_Timing,Expected_Return_Pct,Actual_Strategy_Return_Pct,Passive_Market_Return_Pct,Strategy_Signal
0,DOGEUSDT,0.091740,0.089909,0.093273,T+8,T+20,3.742030,2.034534,1.744064,LONG
1,XRPUSDT,1.311700,1.220502,1.255601,T+6,T+13,2.875791,-0.290442,3.209581,LONG
2,ETHUSDT,2061.270020,2090.061241,2148.154191,T+6,T+21,2.779486,5.532760,9.525685,LONG
3,SOLUSDT,80.650002,109.530168,108.097966,T+1,T+7,1.307587,1.029140,5.306880,HOLD
4,BTCUSDT,67177.000000,67503.899269,66670.572679,T+1,T+7,1.234487,-2.324759,8.655190,HOLD


...

✅ Direct Inference Engine 로드 완료


Evaluating Symbols: 100%|██████████| 5/5 [00:08<00:00,  1.62s/it]

📊 [TFT 모형 기반 전종목 스윙 전략 마스터 요약 리포트]


,Symbol,Current_Price(4/5),Predicted_Entry_Price,Predicted_Exit_Price,Best_Entry_Timing,Best_Exit_Timing,Expected_Return_Pct,Actual_Strategy_Return_Pct,Passive_Market_Return_Pct,Strategy_Signal
0,DOGEUSDT,0.091740,0.089909,0.093273,T+8,T+20,3.742030,2.034534,1.744064,HOLD
1,XRPUSDT,1.311700,1.220502,1.255601,T+6,T+13,2.875791,-0.290442,3.209581,HOLD
2,ETHUSDT,2061.270020,2090.061241,2148.154191,T+6,T+21,2.779486,5.532760,9.525685,HOLD
3,SOLUSDT,80.650002,109.530168,108.097966,T+1,T+7,1.307587,1.029140,5.306880,HOLD
4,BTCUSDT,67177.000000,67503.899269,66670.572679,T+1,T+7,1.234487,-2.324759,8.655190,HOLD



⚠️ 임계치(4.0%)를 넘는 롱/숏 추천 자산이 존재하지 않습니다.


In [61]:
# 62

CHOSEN_SEQ = 42  # 21, 42, 63, 84 중 현재 평가할 폴더의 하이퍼파라미터 입력

completed_outdir = project_path + "outputs/dl/62_2605180617_D128_H64_N32_SEQ42/"
for thres in [0.02,0.03,0.05]:
    profit_simulation(CHOSEN_SEQ,completed_outdir,threshold=thres)

/content/drive/MyDrive/GCY6902_01_TFT/005.regression_final/inference.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  
/content/drive/MyDrive/GCY6902_01_TFT/005.regression_final/inference.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  all_symbols = df['Symbol'].unique()


✅ Direct Inference Engine 로드 완료


Evaluating Symbols: 100%|██████████| 5/5 [00:08<00:00,  1.64s/it]

📊 [TFT 모형 기반 전종목 스윙 전략 마스터 요약 리포트]


,Symbol,Current_Price(4/5),Predicted_Entry_Price,Predicted_Exit_Price,Best_Entry_Timing,Best_Exit_Timing,Expected_Return_Pct,Actual_Strategy_Return_Pct,Passive_Market_Return_Pct,Strategy_Signal
0,XRPUSDT,1.311700,1.200892,1.672997,T+6,T+21,39.312824,0.819187,3.209581,LONG
1,ETHUSDT,2061.270020,2127.811461,2431.542894,T+6,T+12,14.274358,3.762048,9.525685,LONG
2,DOGEUSDT,0.091740,0.084965,0.095062,T+6,T+21,11.883731,1.126765,1.744064,LONG
3,BTCUSDT,67177.000000,66685.357485,71092.017967,T+6,T+12,6.608138,2.582735,8.655190,LONG
4,SOLUSDT,80.650002,111.247713,117.764370,T+6,T+12,5.857790,2.317312,5.306880,LONG


...

✅ Direct Inference Engine 로드 완료


Evaluating Symbols: 100%|██████████| 5/5 [00:08<00:00,  1.64s/it]

📊 [TFT 모형 기반 전종목 스윙 전략 마스터 요약 리포트]


,Symbol,Current_Price(4/5),Predicted_Entry_Price,Predicted_Exit_Price,Best_Entry_Timing,Best_Exit_Timing,Expected_Return_Pct,Actual_Strategy_Return_Pct,Passive_Market_Return_Pct,Strategy_Signal
0,XRPUSDT,1.311700,1.200892,1.672997,T+6,T+21,39.312824,0.819187,3.209581,LONG
1,ETHUSDT,2061.270020,2127.811461,2431.542894,T+6,T+12,14.274358,3.762048,9.525685,LONG
2,DOGEUSDT,0.091740,0.084965,0.095062,T+6,T+21,11.883731,1.126765,1.744064,LONG
3,BTCUSDT,67177.000000,66685.357485,71092.017967,T+6,T+12,6.608138,2.582735,8.655190,LONG
4,SOLUSDT,80.650002,111.247713,117.764370,T+6,T+12,5.857790,2.317312,5.306880,LONG



🎯 [전략 포트폴리오 백테스트 정산 리포트 (Long-Short 통합)]
🔹 선정 포트폴리오 자산 상황:
   - XRPUSDT: LONG (T+6 ➡️ T+21)
   - ETHUSDT: LONG (T+6 ➡️ T+12)
   - DOGEUSDT: LONG (T+6 ➡️ T+21)
   - BTCUSDT: LONG (T+6 ➡️ T+12)
   - SOLUSDT: LONG (T+6 ➡️ T+12)
------------------------------------------------------------------
📈 모델 궤적상 최적 스윙 예상 수익률     : 15.59%
💰 해당 타이밍 실전 진입 시 실제 수익률  : 2.12%
💤 동기간 전종목 단순 매수(long) 후 방치 수익률 : 5.69%
🚀 양방향 타이밍 최적화를 통한 초과 알파  : -3.57%p

💾 대시보드 시각화용 아웃풋 파일 아티팩트 저장 완료 ➡️ 경로: /content/drive/MyDrive/GCY6902_01_TFT/005.regression_final/outputs/dl/62_2605180617_D128_H64_N32_SEQ42/


In [62]:
# 22

CHOSEN_SEQ = 21  # 21, 42, 63, 84 중 현재 평가할 폴더의 하이퍼파라미터 입력

completed_outdir = project_path + "outputs/dl/22_2605180028_D32_H256_N16_SEQ21/"

for thres in [0.02,0.03,0.05]:
    profit_simulation(CHOSEN_SEQ,completed_outdir,threshold=thres)

/content/drive/MyDrive/GCY6902_01_TFT/005.regression_final/inference.py:69: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  
/content/drive/MyDrive/GCY6902_01_TFT/005.regression_final/inference.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  all_symbols = df['Symbol'].unique()


✅ Direct Inference Engine 로드 완료


Evaluating Symbols: 100%|██████████| 5/5 [00:08<00:00,  1.66s/it]

📊 [TFT 모형 기반 전종목 스윙 전략 마스터 요약 리포트]


,Symbol,Current_Price(4/5),Predicted_Entry_Price,Predicted_Exit_Price,Best_Entry_Timing,Best_Exit_Timing,Expected_Return_Pct,Actual_Strategy_Return_Pct,Passive_Market_Return_Pct,Strategy_Signal
0,DOGEUSDT,0.091740,0.088795,0.085974,T+1,T+10,3.177862,-2.692393,1.744064,SHORT
1,ETHUSDT,2061.270020,2061.822385,2109.948872,T+6,T+21,2.334172,5.532760,9.525685,LONG
2,XRPUSDT,1.311700,1.213439,1.188904,T+1,T+21,2.021907,-3.209581,3.209581,SHORT
3,BTCUSDT,67177.000000,66802.526724,65664.750098,T+1,T+17,1.703194,-6.541824,8.655190,HOLD
4,SOLUSDT,80.650002,108.145460,106.764947,T+1,T+7,1.276533,1.029140,5.306880,HOLD


...

✅ Direct Inference Engine 로드 완료


Evaluating Symbols: 100%|██████████| 5/5 [00:08<00:00,  1.65s/it]

📊 [TFT 모형 기반 전종목 스윙 전략 마스터 요약 리포트]


,Symbol,Current_Price(4/5),Predicted_Entry_Price,Predicted_Exit_Price,Best_Entry_Timing,Best_Exit_Timing,Expected_Return_Pct,Actual_Strategy_Return_Pct,Passive_Market_Return_Pct,Strategy_Signal
0,DOGEUSDT,0.091740,0.088795,0.085974,T+1,T+10,3.177862,-2.692393,1.744064,HOLD
1,ETHUSDT,2061.270020,2061.822385,2109.948872,T+6,T+21,2.334172,5.532760,9.525685,HOLD
2,XRPUSDT,1.311700,1.213439,1.188904,T+1,T+21,2.021907,-3.209581,3.209581,HOLD
3,BTCUSDT,67177.000000,66802.526724,65664.750098,T+1,T+17,1.703194,-6.541824,8.655190,HOLD
4,SOLUSDT,80.650002,108.145460,106.764947,T+1,T+7,1.276533,1.029140,5.306880,HOLD



⚠️ 임계치(5.0%)를 넘는 롱/숏 추천 자산이 존재하지 않습니다.
